# AgroSmart Andino — Pipeline Big Data con PySpark
### Parte 4: Procesamiento Distribuido en AWS EMR

**Proyecto:** AgroSmart Andino — Monitoreo Inteligente de Papa Nativa en Huangascar, Yauyos  
**Curso:** Big Data | Maestría en Inteligencia Artificial — UNI  
**Grupo 2:** Anahys Montes, Christian Vizcardo, Cristhian Massa, Freddy Huali, Wilmer Lazaro  

---

## Estructura del Pipeline

| Sección | Descripción |
|---------|-------------|
| 1. Configuración | SparkSession + parámetros S3 |
| 2. ETL | Lectura S3 → limpieza → join → Parquet |
| 3. EDA | Estadísticas descriptivas + correlaciones |
| 4. Índices Agroclimáticos | Helada, tizón tardío, estrés hídrico, ET₀ |
| 5. Machine Learning | Random Forest (clasificación) + GBT (regresión) |
| 6. Exportación | Resultados para dashboard Streamlit + Cassandra |

---
## 1. Configuración del entorno

In [ ]:
# ── Verificar versión de PySpark y Spark ──────────────────────────────────
import sys
print(f'Python  : {sys.version}')

from pyspark.sql import SparkSession
print(f'PySpark : {SparkSession.builder.getOrCreate().version}')

In [ ]:
# ── Instalar librerias al kernel en ejecucion (sin restart) ──────────────
import subprocess, sys

INSTALL_DIR = "/tmp/py_packages"

pkgs = ["pandas", "numpy", "matplotlib", "seaborn"]
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "--target", INSTALL_DIR] + pkgs,
    capture_output=True, text=True
)
if result.returncode != 0:
    print("ERROR al instalar:")
    print(result.stderr[:1000])
else:
    print("Instalacion OK")

# Agregar el directorio al path del kernel actual
if INSTALL_DIR not in sys.path:
    sys.path.insert(0, INSTALL_DIR)

# Verificar
import pandas
print(f"pandas {pandas.__version__} listo")


In [ ]:
# ── Imports principales ───────────────────────────────────────────────────
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, DateType, TimestampType, BooleanType
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, RegressionEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('✓ Imports cargados correctamente')

In [ ]:
# ── SparkSession optimizada para demo en EMR ──────────────────────────────
BUCKET = 'agrosmart-andino-demo'   # ← Cambiar si usas otro nombre
REGION = 'us-east-1'

spark = (
    SparkSession.builder
    .appName('AgroSmart-Andino-Pipeline')
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '4g')
    .config('spark.executor.cores', '2')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '8')   # demo: clúster pequeño
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

# Rutas S3
S3_RAW  = f's3://{BUCKET}/raw'
S3_PROC = f's3://{BUCKET}/processed'
S3_GOLD = f's3://{BUCKET}/gold'

print(f'✓ SparkSession activa: {spark.version}')
print(f'  Master  : {spark.sparkContext.master}')
print(f'  App ID  : {spark.sparkContext.applicationId}')
print(f'  S3 Raw  : {S3_RAW}')
print(f'  S3 Gold : {S3_GOLD}')

---
## 2. ETL — Extracción, Transformación y Carga

In [ ]:
# ── 2.1 Leer dataset climático (NASA POWER) ───────────────────────────────
df_clima_raw = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{S3_RAW}/clima/nasa_power_yauyos.csv')
)

print(f'NASA POWER — filas raw: {df_clima_raw.count():,}')
print(f'Columnas: {df_clima_raw.columns}')
df_clima_raw.printSchema()

In [ ]:
# ── 2.2 Limpiar y tipar dataset climático ────────────────────────────────
df_clima = (
    df_clima_raw
    .withColumn('fecha',        F.to_date('fecha', 'yyyy-MM-dd'))
    .withColumn('temp_max_c',   F.col('temp_max_c').cast(DoubleType()))
    .withColumn('temp_min_c',   F.col('temp_min_c').cast(DoubleType()))
    .withColumn('temp_media_c', F.col('temp_media_c').cast(DoubleType()))
    .withColumn('precipitacion_mm',        F.col('precipitacion_mm').cast(DoubleType()))
    .withColumn('humedad_relativa_pct',     F.col('humedad_relativa_pct').cast(DoubleType()))
    .withColumn('radiacion_solar_mj',       F.col('radiacion_solar_mj').cast(DoubleType()))
    .withColumn('viento_ms',               F.col('viento_ms').cast(DoubleType()))
    .withColumn('riesgo_helada',           F.col('riesgo_helada').cast(IntegerType()))
    .withColumn('dia_lluvioso',            F.col('dia_lluvioso').cast(IntegerType()))
    # Eliminar filas con nulos en variables clave
    .dropna(subset=['fecha', 'temp_media_c', 'precipitacion_mm', 'humedad_relativa_pct'])
    # Filtrar rangos válidos
    .filter(F.col('temp_media_c').between(-20, 40))
    .filter(F.col('precipitacion_mm').between(0, 200))
    .filter(F.col('humedad_relativa_pct').between(0, 100))
)

# Verificar
n = df_clima.count()
print(f'NASA POWER — filas limpias: {n:,}')
df_clima.select('fecha', 'estacion', 'temp_media_c', 'precipitacion_mm',
                'humedad_relativa_pct', 'riesgo_helada').show(5, truncate=False)

In [ ]:
# ── 2.3 Leer y limpiar dataset IoT ───────────────────────────────────────
df_iot_raw = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{S3_RAW}/sensores/sensores_iot.csv')
)

df_iot = (
    df_iot_raw
    .withColumn('timestamp',          F.to_timestamp('timestamp', 'yyyy-MM-dd HH:mm:ss'))
    .withColumn('fecha',              F.to_date('timestamp'))
    .withColumn('hora',               F.hour('timestamp'))
    .withColumn('temp_suelo_c',       F.col('temp_suelo_c').cast(DoubleType()))
    .withColumn('humedad_suelo_pct',  F.col('humedad_suelo_pct').cast(DoubleType()))
    .withColumn('caudal_riego_lmin',  F.col('caudal_riego_lmin').cast(DoubleType()))
    .withColumn('bateria_pct',        F.col('bateria_pct').cast(DoubleType()))
    .dropna(subset=['timestamp', 'temp_suelo_c', 'humedad_suelo_pct'])
    .filter(F.col('temp_suelo_c').between(-5, 30))
    .filter(F.col('humedad_suelo_pct').between(0, 100))
)

print(f'IoT — filas limpias: {df_iot.count():,}')
df_iot.select('timestamp', 'nodo_id', 'temp_suelo_c',
              'humedad_suelo_pct', 'caudal_riego_lmin', 'bateria_pct').show(5, truncate=False)

In [ ]:
# ── 2.4 Leer y limpiar dataset de producción ─────────────────────────────
df_prod_raw = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(f'{S3_RAW}/produccion/produccion_papa_yauyos.csv')
)

df_prod = (
    df_prod_raw
    .withColumn('anio',               F.col('anio').cast(IntegerType()))
    .withColumn('sup_sembrada_ha',    F.col('sup_sembrada_ha').cast(DoubleType()))
    .withColumn('sup_cosechada_ha',   F.col('sup_cosechada_ha').cast(DoubleType()))
    .withColumn('produccion_tm',      F.col('produccion_tm').cast(DoubleType()))
    .withColumn('rendimiento_tm_ha',  F.col('rendimiento_tm_ha').cast(DoubleType()))
    .withColumn('precio_chacra_sol_kg', F.col('precio_chacra_sol_kg').cast(DoubleType()))
    .dropna(subset=['anio', 'rendimiento_tm_ha', 'produccion_tm'])
    .filter(F.col('rendimiento_tm_ha').between(1, 30))
)

print(f'Producción — filas limpias: {df_prod.count():,}')
df_prod.select('anio', 'distrito', 'variedad', 'produccion_tm',
               'rendimiento_tm_ha', 'evento_climatico').show(5, truncate=False)

In [ ]:
# ── 2.5 Agregar IoT por día y nodo (resumir lecturas horarias) ───────────
df_iot_diario = (
    df_iot
    .groupBy('fecha', 'nodo_id', 'parcela', 'altitud_m')
    .agg(
        F.avg('temp_suelo_c').alias('temp_suelo_media'),
        F.min('temp_suelo_c').alias('temp_suelo_min'),
        F.max('temp_suelo_c').alias('temp_suelo_max'),
        F.avg('humedad_suelo_pct').alias('humedad_suelo_media'),
        F.min('humedad_suelo_pct').alias('humedad_suelo_min'),
        F.sum('caudal_riego_lmin').alias('vol_riego_total_l'),
        F.sum(F.col('riego_activo').cast(IntegerType())).alias('horas_riego'),
        F.sum(F.col('alerta_helada_suelo').cast(IntegerType())).alias('horas_alerta_helada'),
        F.sum(F.col('alerta_sequia').cast(IntegerType())).alias('horas_alerta_sequia'),
        F.sum(F.col('alerta_exceso_agua').cast(IntegerType())).alias('horas_alerta_exceso'),
        F.avg('bateria_pct').alias('bateria_media'),
        F.count('*').alias('lecturas_dia'),
    )
    .orderBy('fecha', 'nodo_id')
)

print(f'IoT diario — filas: {df_iot_diario.count():,}')
df_iot_diario.show(5, truncate=False)

In [ ]:
# ── 2.6 Join clima + IoT por fecha (usar Huangascar como estación base) ──
# Filtrar clima solo para la estación Huangascar (referencia central)
df_clima_huan = df_clima.filter(F.col('estacion') == 'Huangascar')

df_join = (
    df_iot_diario
    .join(
        df_clima_huan.select(
            'fecha', 'temp_max_c', 'temp_min_c', 'temp_media_c',
            'precipitacion_mm', 'humedad_relativa_pct', 'radiacion_solar_mj',
            'viento_ms', 'riesgo_helada', 'dia_lluvioso'
        ),
        on='fecha',
        how='left'
    )
    .orderBy('fecha', 'nodo_id')
)

# Rango de fechas del join: 2023-2024 (IoT) — datos clima disponibles para esas fechas
n_join = df_join.count()
print(f'Dataset integrado — filas: {n_join:,}')
print(f'Columnas: {len(df_join.columns)}')

# Verificar nulos post-join
nulos = {c: df_join.filter(F.col(c).isNull()).count() for c in
         ['temp_media_c', 'precipitacion_mm', 'humedad_relativa_pct']}
print(f'Nulos post-join: {nulos}')
df_join.show(3, truncate=False)

In [ ]:
# ── 2.7 Escribir datos procesados como Parquet en S3 ─────────────────────
# Clima limpio
(
    df_clima
    .coalesce(2)
    .write
    .mode('overwrite')
    .parquet(f'{S3_PROC}/clima_limpio/')
)
print('✓ Parquet: clima_limpio')

# IoT diario agregado
(
    df_iot_diario
    .coalesce(2)
    .write
    .mode('overwrite')
    .parquet(f'{S3_PROC}/iot_diario/')
)
print('✓ Parquet: iot_diario')

# Dataset integrado
(
    df_join
    .coalesce(4)
    .write
    .mode('overwrite')
    .parquet(f'{S3_PROC}/dataset_integrado/')
)
print('✓ Parquet: dataset_integrado')

# Producción
(
    df_prod
    .coalesce(1)
    .write
    .mode('overwrite')
    .parquet(f'{S3_PROC}/produccion_limpia/')
)
print('✓ Parquet: produccion_limpia')
print(f'\n  Datos procesados en: {S3_PROC}/')

---
## 3. EDA — Análisis Exploratorio de Datos

In [ ]:
# ── 3.1 Estadísticas descriptivas del clima ───────────────────────────────
print('=== Estadísticas climáticas (Huangascar 2015-2024) ===')
df_clima_huan.select(
    'temp_max_c', 'temp_min_c', 'temp_media_c',
    'precipitacion_mm', 'humedad_relativa_pct', 'radiacion_solar_mj'
).describe().show(truncate=False)

print('=== Días con riesgo de helada por año (Huangascar) ===')
(
    df_clima_huan
    .filter(F.col('riesgo_helada') == 1)
    .groupBy('anio')
    .count()
    .orderBy('anio')
    .show()
)

In [ ]:
# ── 3.2 Estadísticas IoT por nodo ─────────────────────────────────────────
print('=== Estadísticas IoT por nodo (2023-2024) ===')
(
    df_iot
    .groupBy('nodo_id', 'parcela')
    .agg(
        F.count('*').alias('lecturas'),
        F.round(F.avg('temp_suelo_c'), 2).alias('temp_suelo_media'),
        F.round(F.avg('humedad_suelo_pct'), 1).alias('humedad_media'),
        F.sum(F.col('alerta_helada_suelo').cast(IntegerType())).alias('total_alertas_helada'),
        F.sum(F.col('alerta_sequia').cast(IntegerType())).alias('total_alertas_sequia'),
        F.sum(F.col('riego_activo').cast(IntegerType())).alias('total_horas_riego'),
    )
    .orderBy('nodo_id')
    .show(truncate=False)
)

In [ ]:
# ── 3.3 Correlación entre variables climáticas ────────────────────────────
print('=== Matriz de correlación (clima Huangascar) ===')
vars_corr = ['temp_media_c', 'temp_min_c', 'precipitacion_mm',
             'humedad_relativa_pct', 'radiacion_solar_mj', 'viento_ms']

from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

assembler_corr = VectorAssembler(inputCols=vars_corr, outputCol='features_corr')
df_vec = assembler_corr.transform(df_clima_huan.dropna(subset=vars_corr))

matrix = Correlation.corr(df_vec, 'features_corr').head()[0].toArray()
df_corr = pd.DataFrame(matrix, index=vars_corr, columns=vars_corr)

plt.figure(figsize=(8, 6))
mask = np.zeros_like(df_corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(df_corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlación entre variables climáticas\nHuangascar, Yauyos 2015-2024', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/corr_clima.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Gráfico guardado: /tmp/corr_clima.png')

In [ ]:
# ── 3.4 Temperatura mensual promedio (boxplot por mes) ────────────────────
df_clima_pd = (
    df_clima_huan
    .select('mes', 'temp_max_c', 'temp_min_c', 'temp_media_c', 'precipitacion_mm')
    .toPandas()
)

meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Temperatura por mes
data_temp = [df_clima_pd[df_clima_pd['mes'] == m]['temp_media_c'].values for m in range(1, 13)]
axes[0].boxplot(data_temp, labels=meses, patch_artist=True,
                boxprops=dict(facecolor='#A8D5A2', color='#2D6A4F'),
                medianprops=dict(color='#D62828', linewidth=2))
axes[0].axhline(y=2, color='blue', linestyle='--', alpha=0.7, label='Umbral helada (2°C)')
axes[0].set_title('Temperatura Media Mensual — Huangascar', fontsize=11)
axes[0].set_ylabel('Temperatura (°C)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precipitación mensual acumulada
precip_mes = df_clima_pd.groupby('mes')['precipitacion_mm'].mean()
colors = ['#4169E1' if m in [11,12,1,2,3,4] else '#DEB887' for m in range(1, 13)]
axes[1].bar(meses, precip_mes.values, color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('Precipitación Media Mensual — Huangascar', fontsize=11)
axes[1].set_ylabel('Precipitación (mm/día)')
axes[1].grid(True, alpha=0.3, axis='y')
# Leyenda
from matplotlib.patches import Patch
axes[1].legend(handles=[
    Patch(facecolor='#4169E1', label='Temporada lluviosa (Nov-Abr)'),
    Patch(facecolor='#DEB887', label='Temporada seca (May-Oct)'),
], fontsize=9)

plt.suptitle('Análisis Climático Histórico — Huangascar, Yauyos (2015-2024)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/eda_clima.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Gráfico guardado: /tmp/eda_clima.png')

In [ ]:
# ── 3.5 Tendencia anual de rendimiento de papa ────────────────────────────
df_rend_anio = (
    df_prod
    .groupBy('anio')
    .agg(
        F.round(F.avg('rendimiento_tm_ha'), 2).alias('rendimiento_prom'),
        F.round(F.sum('produccion_tm'), 0).alias('produccion_total'),
        F.round(F.sum('sup_sembrada_ha'), 0).alias('ha_sembradas'),
    )
    .orderBy('anio')
    .toPandas()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(df_rend_anio['anio'], df_rend_anio['rendimiento_prom'],
             marker='o', color='#2D6A4F', linewidth=2, markersize=6)
axes[0].fill_between(df_rend_anio['anio'], df_rend_anio['rendimiento_prom'],
                     alpha=0.2, color='#52B788')
axes[0].set_ylabel('Rendimiento (t/ha)', fontsize=10)
axes[0].set_title('Rendimiento Promedio de Papa Nativa — Yauyos (2000-2024)', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Marcar eventos climáticos extremos
eventos = {2010: 'El Niño', 2020: 'COVID\n+Heladas'}
for anio, label in eventos.items():
    val = df_rend_anio[df_rend_anio['anio'] == anio]['rendimiento_prom'].values
    if len(val):
        axes[0].annotate(label, xy=(anio, val[0]),
                         xytext=(anio + 0.5, val[0] + 0.3),
                         fontsize=8, color='red',
                         arrowprops=dict(arrowstyle='->', color='red', lw=1))

axes[1].bar(df_rend_anio['anio'], df_rend_anio['produccion_total'],
            color='#8BC34A', edgecolor='#558B2F', alpha=0.8)
axes[1].set_xlabel('Año', fontsize=10)
axes[1].set_ylabel('Producción total (tm)', fontsize=10)
axes[1].set_title('Producción Total Anual — Papa Nativa Yauyos', fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/eda_produccion.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Gráfico guardado: /tmp/eda_produccion.png')

---
## 4. Índices Agroclimáticos

Se calculan 4 índices para alertas y recomendaciones:

| Índice | Descripción | Umbral crítico |
|--------|-------------|----------------|
| **IHB** | Índice de Helada Bioclimática | Tmin < 2°C |
| **ITT** | Riesgo Tizón Tardío (*Phytophthora infestans*) | HR > 85% y T ∈ [10, 24°C] |
| **IEH** | Índice de Estrés Hídrico | Humedad suelo < 25% |
| **ET₀** | Evapotranspiración de referencia (Penman-Monteith simplificado) | — |

In [ ]:
# ── 4.1 Calcular índices agroclimáticos sobre dataset integrado ───────────
df_indices = (
    df_join
    # IHB — Índice Helada Bioclimática (0-3 según severidad)
    .withColumn('ihb',
        F.when(F.col('temp_min_c') < -4, F.lit(3))        # helada severa
         .when(F.col('temp_min_c') < 0,  F.lit(2))        # helada moderada
         .when(F.col('temp_min_c') < 2,  F.lit(1))        # riesgo helada
         .otherwise(F.lit(0))                              # sin riesgo
    )
    # IHB en suelo (temperatura de suelo)
    .withColumn('ihb_suelo',
        F.when(F.col('temp_suelo_min') < 0, F.lit(2))
         .when(F.col('temp_suelo_min') < 2, F.lit(1))
         .otherwise(F.lit(0))
    )
    # ITT — Riesgo Tizón Tardío (Phytophthora infestans)
    # Alta montaña (>3500m): usar temp_max en lugar de temp_media porque
    # el hongo actúa durante las horas cálidas del día (no la media de 24h).
    # Umbrales ajustados a condiciones alto-andinas de Yauyos.
    .withColumn('cond_temp_tizon',
        F.col('temp_max_c').between(10, 25).cast(IntegerType())
    )
    .withColumn('cond_hr_tizon',
        (F.col('humedad_relativa_pct') > 70).cast(IntegerType())
    )
    .withColumn('itt',
        F.when(
            (F.col('humedad_relativa_pct') > 80) &
            F.col('temp_max_c').between(12, 22),
            F.lit(3)  # riesgo muy alto (optimo para P. infestans)
        ).when(
            (F.col('humedad_relativa_pct') > 75) &
            F.col('temp_max_c').between(10, 25),
            F.lit(2)  # riesgo alto
        ).when(
            (F.col('humedad_relativa_pct') > 70) &
            F.col('temp_max_c').between(10, 25),
            F.lit(1)  # riesgo moderado
        ).otherwise(F.lit(0))
    )
    # IEH — Índice de Estrés Hídrico en suelo
    .withColumn('ieh',
        F.when(F.col('humedad_suelo_min') < 15, F.lit(3))  # estres severo
         .when(F.col('humedad_suelo_min') < 25, F.lit(2))  # estres moderado
         .when(F.col('humedad_suelo_min') < 35, F.lit(1))  # estres leve
         .otherwise(F.lit(0))
    )
    # ET0 simplificada (Hargreaves-Samani): ET0 = 0.0023*(Tmed+17.8)*(Tmax-Tmin)^0.5*Ra
    # Ra = radiacion_solar_mj (en MJ/m2/dia)
    .withColumn('et0_mm',
        F.round(
            F.lit(0.0023) *
            (F.col('temp_media_c') + F.lit(17.8)) *
            F.pow(F.col('temp_max_c') - F.col('temp_min_c'), 0.5) *
            F.col('radiacion_solar_mj'),
            2
        )
    )
    # Alerta general (nivel maximo de cualquier indice)
    .withColumn('nivel_alerta',
        F.greatest('ihb', 'itt', 'ieh')
    )
    .withColumn('alerta_critica',
        (F.col('nivel_alerta') >= 2).cast(IntegerType())
    )
)

print(f'Dataset con indices - filas: {df_indices.count():,}')
print()

# Resumen de alertas
print('=== Resumen de alertas por nodo ===')
(
    df_indices
    .groupBy('nodo_id')
    .agg(
        F.count('*').alias('dias'),
        F.sum(F.when(F.col('ihb') > 0, 1).otherwise(0)).alias('dias_riesgo_helada'),
        F.sum(F.when(F.col('itt') >= 2, 1).otherwise(0)).alias('dias_riesgo_tizon'),
        F.sum(F.when(F.col('ieh') >= 2, 1).otherwise(0)).alias('dias_estres_hidrico'),
        F.sum('alerta_critica').alias('dias_alerta_critica'),
        F.round(F.avg('et0_mm'), 2).alias('et0_promedio_mm'),
    )
    .orderBy('nodo_id')
    .show(truncate=False)
)


In [ ]:
# ── 4.2 Visualización de índices en el tiempo ─────────────────────────────
# Tomar nodo central (NODO-002) para visualización
df_viz = (
    df_indices
    .filter(F.col('nodo_id') == 'NODO-002')
    .select('fecha', 'temp_min_c', 'humedad_relativa_pct',
            'humedad_suelo_media', 'ihb', 'itt', 'ieh', 'et0_mm')
    .orderBy('fecha')
    .toPandas()
)
df_viz['fecha'] = pd.to_datetime(df_viz['fecha'])

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# IHB
axes[0].fill_between(df_viz['fecha'], df_viz['ihb'], step='mid',
                     color='#3A86FF', alpha=0.7)
axes[0].set_ylabel('IHB\n(0-3)', fontsize=9)
axes[0].set_title('Índices Agroclimáticos — NODO-002, Parcela Media (2023-2024)', fontsize=11)
axes[0].set_ylim(0, 3.5)
axes[0].axhline(2, color='red', linestyle='--', alpha=0.5, lw=0.8)
axes[0].grid(True, alpha=0.2)

# ITT
colors_itt = df_viz['itt'].map({0: '#90EE90', 1: '#FFA500', 2: '#FF6347', 3: '#8B0000'})
axes[1].bar(df_viz['fecha'], df_viz['itt'], color=colors_itt, width=1)
axes[1].set_ylabel('ITT\n(0-3)', fontsize=9)
axes[1].set_ylim(0, 3.5)
axes[1].grid(True, alpha=0.2)

# IEH
axes[2].fill_between(df_viz['fecha'], df_viz['ieh'], step='mid',
                     color='#FF9E00', alpha=0.7)
axes[2].set_ylabel('IEH\n(0-3)', fontsize=9)
axes[2].set_ylim(0, 3.5)
axes[2].axhline(2, color='red', linestyle='--', alpha=0.5, lw=0.8)
axes[2].grid(True, alpha=0.2)

# ET₀
axes[3].plot(df_viz['fecha'], df_viz['et0_mm'], color='#06D6A0', linewidth=0.8, alpha=0.9)
axes[3].fill_between(df_viz['fecha'], df_viz['et0_mm'], alpha=0.2, color='#06D6A0')
axes[3].set_ylabel('ET₀ (mm/día)', fontsize=9)
axes[3].set_xlabel('Fecha', fontsize=10)
axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[3].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
axes[3].grid(True, alpha=0.2)

plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.savefig('/tmp/indices_agroclima.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Gráfico guardado: /tmp/indices_agroclima.png')

---
## 5. Machine Learning

### Modelo 1 — Clasificador de Riesgo de Tizón Tardío (Random Forest)
**Variable objetivo:** `riesgo_tizon_alto` (binaria: ITT ≥ 2)

### Modelo 2 — Predictor de Rendimiento de Papa (Gradient Boosted Trees)
**Variable objetivo:** `rendimiento_tm_ha` (continua)

In [ ]:
# ── 5.1 Feature engineering para ML ──────────────────────────────────────
# Umbral dinámico: percentil 70 del ITT garantiza ~30% clase 1 (alto riesgo)
itt_p70 = df_indices.approxQuantile('itt', [0.70], 0.01)[0]
print(f'ITT percentil 70 (umbral alto riesgo): {itt_p70:.4f}')

df_ml_base = (
    df_indices
    .withColumn('riesgo_tizon_alto',
                (F.col('itt') >= F.lit(itt_p70)).cast(IntegerType()))
    .withColumn('mes',     F.month('fecha'))
    .withColumn('semana',  F.weekofyear('fecha'))
    # Variables de lag (clima del día anterior)
    .withColumn('temp_min_lag1',
        F.lag('temp_min_c', 1).over(
            Window.partitionBy('nodo_id').orderBy('fecha')
        )
    )
    .withColumn('hr_lag1',
        F.lag('humedad_relativa_pct', 1).over(
            Window.partitionBy('nodo_id').orderBy('fecha')
        )
    )
    # Media móvil 7 días de temperatura mínima
    .withColumn('temp_min_ma7',
        F.avg('temp_min_c').over(
            Window.partitionBy('nodo_id')
                  .orderBy('fecha')
                  .rowsBetween(-6, 0)
        )
    )
    # Media móvil 7 días de humedad relativa
    .withColumn('hr_ma7',
        F.avg('humedad_relativa_pct').over(
            Window.partitionBy('nodo_id')
                  .orderBy('fecha')
                  .rowsBetween(-6, 0)
        )
    )
    .dropna(subset=['temp_min_lag1', 'hr_lag1', 'temp_min_ma7', 'hr_ma7'])
)

print(f'Dataset ML - filas: {df_ml_base.count():,}')
df_ml_base.groupBy('riesgo_tizon_alto').count().show()


In [ ]:
# ── 5.2 Random Forest — Clasificación de Riesgo de Tizón Tardío ──────────
FEATURES_RF = [
    'temp_media_c', 'temp_min_c', 'temp_max_c',
    'humedad_relativa_pct', 'precipitacion_mm', 'radiacion_solar_mj',
    'viento_ms', 'temp_suelo_media', 'humedad_suelo_media',
    'temp_min_lag1', 'hr_lag1', 'temp_min_ma7', 'hr_ma7',
    'mes', 'altitud_m', 'et0_mm',
]
TARGET_RF = 'riesgo_tizon_alto'

# Split train/test (80/20 por fecha para evitar data leakage)
fechas = df_ml_base.select('fecha').distinct().orderBy('fecha').collect()
fecha_split = fechas[int(len(fechas) * 0.8)]['fecha']
print(f'Split date: {fecha_split}  (80% train | 20% test)')

df_train = df_ml_base.filter(F.col('fecha') < fecha_split)
df_test  = df_ml_base.filter(F.col('fecha') >= fecha_split)
print(f'Train: {df_train.count():,}  |  Test: {df_test.count():,}')

# Pipeline ML
assembler_rf = VectorAssembler(inputCols=FEATURES_RF, outputCol='features',
                               handleInvalid='skip')
rf = RandomForestClassifier(
    featuresCol='features',
    labelCol=TARGET_RF,
    numTrees=100,
    maxDepth=8,
    seed=42,
    featureSubsetStrategy='sqrt',
)
pipeline_rf = Pipeline(stages=[assembler_rf, rf])

# Entrenar
print('\nEntrenando Random Forest...')
model_rf = pipeline_rf.fit(df_train)
print('OK Modelo entrenado')

# Evaluar
preds_rf = model_rf.transform(df_test)

# Verificar distribucion de clases en test
dist_test = preds_rf.groupBy(TARGET_RF).count().collect()
clases_test = {r[TARGET_RF]: r['count'] for r in dist_test}
print(f'Clases en test: {clases_test}')

eval_auc = BinaryClassificationEvaluator(labelCol=TARGET_RF, metricName='areaUnderROC')
eval_pr  = BinaryClassificationEvaluator(labelCol=TARGET_RF, metricName='areaUnderPR')

# Guard: BinaryClassificationEvaluator requiere 2 clases en rawPrediction
if len(clases_test) >= 2:
    auc = eval_auc.evaluate(preds_rf)
    pr  = eval_pr.evaluate(preds_rf)
else:
    print(f'  AVISO: solo 1 clase en test ({clases_test}) - AUC no calculable')
    auc, pr = 0.5, 0.5

# Accuracy manual
total     = preds_rf.count()
correctos = preds_rf.filter(F.col('prediction') == F.col(TARGET_RF)).count()
accuracy  = correctos / total

print(f'\n=== Metricas Random Forest - Riesgo Tizon Tardio ===')
print(f'  AUC-ROC   : {auc:.4f}')
print(f'  AUC-PR    : {pr:.4f}')
print(f'  Accuracy  : {accuracy:.4f}')

# Importancia de features
rf_model     = model_rf.stages[-1]
importancias = sorted(
    zip(FEATURES_RF, rf_model.featureImportances.toArray()),
    key=lambda x: x[1], reverse=True
)
print(f'\n  Top-8 features mas importantes:')
for feat, imp in importancias[:8]:
    bar = '#' * int(imp * 100)
    print(f'    {feat:30s}: {imp:.4f}  {bar}')


In [ ]:
# ── 5.3 GBT — Predicción de Rendimiento de Papa ───────────────────────────
# Preparar dataset de producción con clima anual agregado
df_clima_anio = (
    df_clima_huan
    .groupBy('anio')
    .agg(
        F.avg('temp_media_c').alias('temp_media_anio'),
        F.avg('temp_min_c').alias('temp_min_anio'),
        F.sum('precipitacion_mm').alias('precip_anual_mm'),
        F.avg('humedad_relativa_pct').alias('hr_media_anio'),
        F.sum('riesgo_helada').alias('dias_helada_anio'),
        F.avg('radiacion_solar_mj').alias('rad_media_anio'),
    )
)

# Indexar variables categóricas
indexer_var  = StringIndexer(inputCol='variedad',        outputCol='variedad_idx',  handleInvalid='keep')
indexer_dist = StringIndexer(inputCol='distrito',        outputCol='distrito_idx',  handleInvalid='keep')
indexer_ev   = StringIndexer(inputCol='evento_climatico', outputCol='evento_idx',   handleInvalid='keep')

df_prod_ml = (
    df_prod
    .join(df_clima_anio, on='anio', how='left')
    .dropna()
)

FEATURES_GBT = [
    'anio', 'sup_sembrada_ha', 'variedad_idx', 'distrito_idx', 'evento_idx',
    'temp_media_anio', 'temp_min_anio', 'precip_anual_mm',
    'hr_media_anio', 'dias_helada_anio', 'rad_media_anio',
]
TARGET_GBT = 'rendimiento_tm_ha'

# Split temporal
df_train_gbt = df_prod_ml.filter(F.col('anio') <= 2020)
df_test_gbt  = df_prod_ml.filter(F.col('anio') > 2020)
print(f'Train GBT: {df_train_gbt.count():,}  |  Test GBT: {df_test_gbt.count():,}')

assembler_gbt = VectorAssembler(inputCols=FEATURES_GBT, outputCol='features',
                                handleInvalid='skip')
gbt = GBTRegressor(
    featuresCol='features',
    labelCol=TARGET_GBT,
    maxIter=100,
    maxDepth=5,
    stepSize=0.05,
    seed=42,
)

pipeline_gbt = Pipeline(stages=[indexer_var, indexer_dist, indexer_ev,
                                 assembler_gbt, gbt])

print('\nEntrenando GBT Regressor...')
model_gbt = pipeline_gbt.fit(df_train_gbt)
print('✓ Modelo entrenado')

preds_gbt = model_gbt.transform(df_test_gbt)

eval_rmse = RegressionEvaluator(labelCol=TARGET_GBT, metricName='rmse')
eval_mae  = RegressionEvaluator(labelCol=TARGET_GBT, metricName='mae')
eval_r2   = RegressionEvaluator(labelCol=TARGET_GBT, metricName='r2')

rmse = eval_rmse.evaluate(preds_gbt)
mae  = eval_mae.evaluate(preds_gbt)
r2   = eval_r2.evaluate(preds_gbt)

print(f'\n=== Métricas GBT — Predicción de Rendimiento ===')
print(f'  RMSE : {rmse:.4f} t/ha')
print(f'  MAE  : {mae:.4f} t/ha')
print(f'  R²   : {r2:.4f}')

# Comparar predicciones vs valores reales
print('\n  Muestra predicciones (test 2021-2024):')
preds_gbt.select(
    'anio', 'variedad', 'distrito',
    F.round(TARGET_GBT, 2).alias('real_tm_ha'),
    F.round('prediction', 2).alias('pred_tm_ha')
).show(10, truncate=False)

In [ ]:
# ── 5.4 Visualización: feature importance RF + predicciones GBT ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Feature importance Random Forest
feats_top10 = importancias[:10]
nombres = [f[0] for f in feats_top10]
valores = [f[1] for f in feats_top10]
colores = ['#D62828' if v > 0.1 else '#F77F00' if v > 0.05 else '#588157'
           for v in valores]
axes[0].barh(nombres[::-1], valores[::-1], color=colores[::-1], edgecolor='white')
axes[0].set_xlabel('Importancia', fontsize=10)
axes[0].set_title('Random Forest — Importancia de Variables\n(Riesgo Tizón Tardío)', fontsize=11)
axes[0].grid(True, alpha=0.3, axis='x')

# GBT: Real vs Predicho
df_preds_pd = preds_gbt.select(TARGET_GBT, 'prediction').toPandas()
axes[1].scatter(df_preds_pd[TARGET_GBT], df_preds_pd['prediction'],
                alpha=0.5, color='#2D6A4F', s=30, edgecolors='white', lw=0.3)
lim = [df_preds_pd[TARGET_GBT].min() - 0.5, df_preds_pd[TARGET_GBT].max() + 0.5]
axes[1].plot(lim, lim, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[1].set_xlabel('Rendimiento Real (t/ha)', fontsize=10)
axes[1].set_ylabel('Rendimiento Predicho (t/ha)', fontsize=10)
axes[1].set_title(f'GBT — Real vs Predicho\n(R² = {r2:.3f}, RMSE = {rmse:.3f} t/ha)', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('AgroSmart Andino — Resultados de Machine Learning', fontsize=13,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/ml_resultados.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Gráfico guardado: /tmp/ml_resultados.png')

---
## 6. Exportación — Resultados para Dashboard y Cassandra

In [ ]:
# ── 6.1 Guardar modelos ML en S3 ─────────────────────────────────────────
model_rf.write().overwrite().save(f'{S3_GOLD}/modelos/random_forest_tizon')
print('✓ Modelo RF guardado')

model_gbt.write().overwrite().save(f'{S3_GOLD}/modelos/gbt_rendimiento')
print('✓ Modelo GBT guardado')

In [ ]:
# ── 6.2 Exportar tabla de lecturas climaticas (→ Cassandra) ───────────────
df_export_clima = (
    df_clima_huan
    .select(
        'fecha', 'estacion',
        F.round('temp_max_c', 2).alias('temp_max_c'),
        F.round('temp_min_c', 2).alias('temp_min_c'),
        F.round('temp_media_c', 2).alias('temp_media_c'),
        F.round('precipitacion_mm', 2).alias('precipitacion_mm'),
        F.round('humedad_relativa_pct', 1).alias('humedad_relativa_pct'),
        F.round('radiacion_solar_mj', 3).alias('radiacion_solar_mj'),
        F.round('viento_ms', 2).alias('viento_ms'),
        'riesgo_helada', 'anio', 'mes'
    )
    .orderBy('fecha')
)

# Parquet para S3
(
    df_export_clima
    .coalesce(1)
    .write.mode('overwrite')
    .parquet(f'{S3_GOLD}/lecturas_climaticas/')
)

# CSV para dashboard local
(
    df_export_clima
    .coalesce(1)
    .write.mode('overwrite')
    .option('header', 'true')
    .csv(f'{S3_GOLD}/csv/lecturas_climaticas/')
)
print('✓ Exportado: lecturas_climaticas')

In [ ]:
# ── 6.3 Exportar alertas activas (últimas 30 días — para Cassandra) ───────
from datetime import datetime, timedelta

# Definir fecha de corte (últimas lecturas disponibles)
fecha_max = df_indices.agg(F.max('fecha')).collect()[0][0]
fecha_min_alertas = fecha_max - timedelta(days=30)
print(f'Rango alertas: {fecha_min_alertas} → {fecha_max}')

df_alertas = (
    df_indices
    .filter(F.col('fecha') >= F.lit(fecha_min_alertas.strftime('%Y-%m-%d')))
    .filter(F.col('nivel_alerta') >= 1)
    .select(
        'fecha', 'nodo_id', 'parcela',
        'ihb', 'itt', 'ieh', 'nivel_alerta', 'alerta_critica',
        F.round('temp_min_c', 1).alias('temp_min_c'),
        F.round('humedad_relativa_pct', 1).alias('humedad_relativa_pct'),
        F.round('humedad_suelo_media', 1).alias('humedad_suelo_media'),
        # Tipo de alerta dominante
        F.when(F.col('ihb') == F.greatest('ihb','itt','ieh'), 'HELADA')
         .when(F.col('itt') == F.greatest('ihb','itt','ieh'), 'TIZON_TARDIO')
         .otherwise('ESTRES_HIDRICO').alias('tipo_alerta'),
    )
    .orderBy('fecha', 'nodo_id')
)

n_alertas = df_alertas.count()
print(f'Alertas últimos 30 días: {n_alertas:,}')
df_alertas.show(10, truncate=False)

(
    df_alertas
    .coalesce(1)
    .write.mode('overwrite')
    .option('header', 'true')
    .csv(f'{S3_GOLD}/csv/alertas_activas/')
)
print('✓ Exportado: alertas_activas')

In [ ]:
# ── 6.4 Exportar recomendaciones de riego ────────────────────────────────
df_riego = (
    df_indices
    .filter(F.col('fecha') >= F.lit(fecha_min_alertas.strftime('%Y-%m-%d')))
    .select(
        'fecha', 'nodo_id', 'parcela', 'altitud_m',
        F.round('et0_mm', 2).alias('et0_mm'),
        F.round('precipitacion_mm', 2).alias('precipitacion_mm'),
        F.round('humedad_suelo_media', 1).alias('humedad_suelo_pct'),
        F.round('vol_riego_total_l', 0).alias('vol_riego_aplicado_l'),
        # Déficit hídrico = ET₀ - precipitación (en mm convertido a litros/m²)
        F.greatest(
            F.round(F.col('et0_mm') - F.col('precipitacion_mm'), 2),
            F.lit(0.0)
        ).alias('deficit_hidrico_mm'),
        # Recomendación de riego
        F.when(
            (F.col('humedad_suelo_media') < 30) | (F.col('ieh') >= 2),
            F.lit('RIEGO_URGENTE')
        ).when(
            (F.col('humedad_suelo_media') < 45) &
            (F.col('et0_mm') > F.col('precipitacion_mm')),
            F.lit('RIEGO_RECOMENDADO')
        ).when(
            F.col('humedad_suelo_media') > 70,
            F.lit('SUSPENDER_RIEGO')
        ).otherwise(F.lit('NORMAL'))
        .alias('recomendacion'),
        'horas_riego',
    )
    .orderBy('fecha', 'nodo_id')
)

print(f'Recomendaciones de riego — filas: {df_riego.count():,}')
print('\nDistribución de recomendaciones:')
df_riego.groupBy('recomendacion').count().show()
df_riego.show(8, truncate=False)

(
    df_riego
    .coalesce(1)
    .write.mode('overwrite')
    .option('header', 'true')
    .csv(f'{S3_GOLD}/csv/recomendaciones_riego/')
)
print('✓ Exportado: recomendaciones_riego')

In [ ]:
# ── 6.5 Exportar predicciones de rendimiento (→ Cassandra) ───────────────
# Generar predicciones para 2025 (escenarios)
escenarios = spark.createDataFrame([
    (2025, 'Huangascar', 'Papa nativa (mix)', 'normal',   55.0, 11.5, 9.2, 3.8, 520.0, 68.0, 14, 18.5),
    (2025, 'Huangascar', 'Huayro',            'normal',   42.0, 11.5, 9.2, 3.8, 520.0, 68.0, 14, 18.5),
    (2025, 'Huangascar', 'Peruanita',         'normal',   38.0, 11.5, 9.2, 3.8, 520.0, 68.0, 14, 18.5),
    (2025, 'Huangascar', 'Papa nativa (mix)', 'helada',   55.0, 10.8, 8.5, 3.2, 510.0, 65.0, 28, 18.5),
    (2025, 'Huangascar', 'Papa nativa (mix)', 'el_nino',  55.0, 12.5, 9.8, 4.5, 640.0, 72.0,  8, 19.2),
], [
    'anio', 'distrito', 'variedad', 'evento_climatico',
    'sup_sembrada_ha',
    'temp_media_anio', 'temp_min_anio', 'precip_anual_mm_div100',
    'precip_anual_mm', 'hr_media_anio', 'dias_helada_anio', 'rad_media_anio',
])
escenarios = escenarios.drop('precip_anual_mm_div100')

preds_2025 = model_gbt.transform(escenarios)

df_pred_export = preds_2025.select(
    'anio', 'distrito', 'variedad', 'evento_climatico',
    'sup_sembrada_ha',
    F.round('prediction', 2).alias('rendimiento_pred_tm_ha'),
    F.round(F.col('prediction') * F.col('sup_sembrada_ha') * F.lit(0.95), 0)
     .alias('produccion_estimada_tm'),
    F.current_timestamp().alias('fecha_prediccion'),
)

print('=== Predicciones de rendimiento 2025 ===')
df_pred_export.show(truncate=False)

(
    df_pred_export
    .coalesce(1)
    .write.mode('overwrite')
    .option('header', 'true')
    .csv(f'{S3_GOLD}/csv/predicciones_rendimiento/')
)
print('✓ Exportado: predicciones_rendimiento')

In [ ]:
# ── 6.6 Guardar gráficos en S3 ────────────────────────────────────────────
import subprocess

graficos = [
    ('/tmp/corr_clima.png',        f's3://{BUCKET}/gold/graficos/corr_clima.png'),
    ('/tmp/eda_clima.png',         f's3://{BUCKET}/gold/graficos/eda_clima.png'),
    ('/tmp/eda_produccion.png',    f's3://{BUCKET}/gold/graficos/eda_produccion.png'),
    ('/tmp/indices_agroclima.png', f's3://{BUCKET}/gold/graficos/indices_agroclima.png'),
    ('/tmp/ml_resultados.png',     f's3://{BUCKET}/gold/graficos/ml_resultados.png'),
]

for local, s3_dest in graficos:
    try:
        result = subprocess.run(
            ['aws', 's3', 'cp', local, s3_dest],
            capture_output=True, text=True, timeout=30
        )
        if result.returncode == 0:
            print(f'✓ {s3_dest.split("/")[-1]}')
        else:
            print(f'✗ Error: {result.stderr.strip()}')
    except Exception as e:
        print(f'✗ {local}: {e}')

In [ ]:
# ── 6.7 Descargar CSVs Gold a carpeta local (para dashboard Streamlit) ────
# Ejecutar en la instancia EMR, luego copiar a laptop
import subprocess
import os

LOCAL_GOLD = '/tmp/agrosmart_gold'
os.makedirs(LOCAL_GOLD, exist_ok=True)

tablas = [
    'lecturas_climaticas',
    'alertas_activas',
    'recomendaciones_riego',
    'predicciones_rendimiento',
]

for tabla in tablas:
    s3_src   = f's3://{BUCKET}/gold/csv/{tabla}/'
    local_dst = f'{LOCAL_GOLD}/{tabla}/'
    result = subprocess.run(
        ['aws', 's3', 'sync', s3_src, local_dst, '--exclude', '*', '--include', '*.csv'],
        capture_output=True, text=True, timeout=60
    )
    if result.returncode == 0:
        # Renombrar el CSV único generado por Spark
        csvs = [f for f in os.listdir(local_dst) if f.endswith('.csv')]
        if csvs:
            os.rename(
                os.path.join(local_dst, csvs[0]),
                os.path.join(LOCAL_GOLD, f'{tabla}.csv')
            )
        print(f'✓ {tabla}.csv')
    else:
        print(f'✗ {tabla}: {result.stderr[:100]}')

print(f'\n  Archivos listos en: {LOCAL_GOLD}/')
print('  → Cópialos a tu laptop (carpeta data/gold/) para el dashboard Streamlit')
print()
print('  Desde tu laptop (PowerShell):')
print(f'    scp -i <key.pem> hadoop@<MASTER_DNS>:/tmp/agrosmart_gold/*.csv')
print(f'    data/gold/  (dentro del proyecto agrosmart_andino/)')

---
## 7. Resumen del Pipeline

In [ ]:
# ── 7.1 Resumen ejecutivo del pipeline ───────────────────────────────────
print('=' * 65)
print('  AgroSmart Andino — Resumen del Pipeline Big Data')
print('=' * 65)
print()
print('  ETL')
print(f'    NASA POWER (clima)      : {df_clima.count():>8,} registros')
print(f'    Sensores IoT (horario)  : {df_iot.count():>8,} registros')
print(f'    IoT diario agregado     : {df_iot_diario.count():>8,} registros')
print(f'    Producción MIDAGRI      : {df_prod.count():>8,} registros')
print(f'    Dataset integrado       : {df_join.count():>8,} registros')
print()
print('  ÍNDICES CALCULADOS')
dias_helada = df_indices.filter(F.col('ihb') > 0).count()
dias_tizon  = df_indices.filter(F.col('itt') >= 2).count()
dias_estres = df_indices.filter(F.col('ieh') >= 2).count()
print(f'    Días riesgo helada      : {dias_helada:>8,}')
print(f'    Días riesgo tizón       : {dias_tizon:>8,}')
print(f'    Días estrés hídrico     : {dias_estres:>8,}')
print()
print('  MACHINE LEARNING')
print(f'    RF Tizón Tardío  — AUC-ROC : {auc:.4f}')
print(f'    RF Tizón Tardío  — Accuracy: {accuracy:.4f}')
print(f'    GBT Rendimiento  — RMSE    : {rmse:.4f} t/ha')
print(f'    GBT Rendimiento  — R²      : {r2:.4f}')
print()
print('  ARTEFACTOS EN S3')
print(f'    s3://{BUCKET}/processed/  → Parquet limpio (4 tablas)')
print(f'    s3://{BUCKET}/gold/csv/   → CSVs para dashboard (4 tablas)')
print(f'    s3://{BUCKET}/gold/modelos/ → Modelos ML guardados')
print(f'    s3://{BUCKET}/gold/graficos/ → Gráficos PNG (5 imágenes)')
print()
print('  PRÓXIMO PASO — PARTE 5')
print('    Cargar CSVs Gold → Apache Cassandra (Cloud9)')
print('    Script: scripts/cassandra/07_create_schema.cql')
print('    Script: scripts/cassandra/08_load_data.py')
print('=' * 65)

In [ ]:
# ── 7.2 Cerrar SparkSession ───────────────────────────────────────────────
spark.stop()
print('✓ SparkSession cerrada')